# Composite Ranking - Optimized Method 2 (Shared-s DirMult)

Combines:

- DirMult LRT padj (significance, omnibus)
- Δψ (effect size, signed Wasserstein-1 on rank space)
- ρ_W (gene-level Spearman of W_cell vs pseudotime on graph-smoothed data; replaces previous per-site `best_rho`/`mean_abs_rho`)
- NMI (non-monotonicity diagnostic, carried through from delta_psi script)
- entropy_rho (Spearman of per-cell Shannon entropy of UTR usage vs pseudotime, concentration vs spread; complementary diagnostic, not in the composite formula)

Composite weights: 50% sig / 25% effect / 25% concordance (entropy_rho is excluded from the composite: it is a separate dimension that does not reduce to the same magnitude scale).


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026/03-optimized_method_2")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib_venn import venn3, venn2
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
    }
)

BASEPATH = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling"
)

# Domain-informed weights (same rationale as method 1 optimized ranking):
# Prioritize statistical significance to prevent effect-size-driven rankings
# where nearly all enriched genes already have padj << 0.05.
W_PADJ = 0.50  # Statistical significance (DirMult LRT)
W_DPSI = 0.25  # Effect size (|delta_psi|)
W_RHO = 0.25  # Monotonic trend (mean |rho|, corrected for selection bias)

## Load data


In [ ]:
# -- Method 2 DirMult LRT results (shared-s, conditional MLE, bootstrap) ------
dm = pd.read_csv(
    BASEPATH / "DirMult_optimized/dirmult_regression_optimized2_joint_df5.csv"
)
dm = dm.rename(columns={"utr_name": "gene_id"})
print(f"DirMult LRT (method 2): {dm.shape}")

# -- Delta-psi + NMI (model-derived effect size) -------------------------------
dp = pd.read_csv(BASEPATH / "DirMult/delta_psi_per_gene_df5_q0.1-0.9.csv")
print(f"Delta psi: {dp.shape}")

# -- Gene-level ρ_W and entropy_rho (graph-smoothed, model-independent) -------
# W_spearman CSV columns: gene_id, gene_name, n_sites, n_cells,
#   rho_W, pvalue_W, padj_BH_W, abs_rho_W,
#   entropy_rho, pvalue_H, padj_BH_H, abs_entropy_rho.
# Replaces the previous per-site Spearman aggregation (best_rho / mean_abs_rho).
# See 05_260504-genelevel-W_spearman_graphsmoothed.ipynb.
rho_w = pd.read_csv(
    BASEPATH / "correlation_analyses_proportions/utr_pseudotime_W_spearman_per_gene.csv"
)
print(f"ρ_W + entropy_rho per-gene: {rho_w.shape}")

## p-value handling

Method 2 provides three p-value flavours:

| Column          | Source                         | Available       | Issue                                |
| --------------- | ------------------------------ | --------------- | ------------------------------------ |
| `p_value_chisq` | Conditional LRT, chi-sq approx | All 8,057 genes | None - full resolution (min ~7e-257) |
| `p_value_boot`  | Parametric bootstrap (B=199)   | 3,880 genes     | Floor at 1/(B+1) = 0.005             |
| `p_value`       | boot if available, else chisq  | All             | Mixes two scales                     |

We recompute BH adjustment from `p_value_chisq` only, on converged genes.


In [ ]:
# Rename the pre-computed padj so it doesn't collide
dm = dm.rename(columns={"padj": "padj_mixed"})

# --- Filter non-converged genes ------------------------------------------------
n_total = len(dm)
converged_mask = dm["converged"] == True
n_converged = converged_mask.sum()
n_nonconv = n_total - n_converged
print(f"Total genes: {n_total:,}")
print(f"Converged:   {n_converged:,}  ({n_converged/n_total*100:.1f}%)")
print(f"Non-converged: {n_nonconv:,}  ({n_nonconv/n_total*100:.1f}%)")

# --- Recompute padj_chisq on converged subset ----------------------------------
dm["padj_chisq"] = np.nan
conv_idx = dm.index[converged_mask]
p_chisq_conv = dm.loc[conv_idx, "p_value_chisq"].values
_, padj_chisq_vals, _, _ = multipletests(p_chisq_conv, method="fdr_bh")
dm.loc[conv_idx, "padj_chisq"] = padj_chisq_vals

# --- Compute padj_boot on bootstrap subset (for comparison) --------------------
dm["padj_boot"] = np.nan
boot_mask = dm["p_value_boot"].notna() & converged_mask
if boot_mask.sum() > 0:
    boot_idx = dm.index[boot_mask]
    _, padj_boot_vals, _, _ = multipletests(
        dm.loc[boot_idx, "p_value_boot"].values, method="fdr_bh"
    )
    dm.loc[boot_idx, "padj_boot"] = padj_boot_vals

# --- Set primary padj to padj_chisq --------------------------------------------
dm["padj"] = dm["padj_chisq"]

# --- Discovery counts ----------------------------------------------------------
print(f"\nDiscovery counts (padj < 0.05):")
print(f"  padj_chisq: {(dm['padj_chisq'] < 0.05).sum():,}")
print(f"  padj_mixed: {(dm['padj_mixed'] < 0.05).sum():,}")
print(
    f"  padj_boot:  {(dm['padj_boot'].dropna() < 0.05).sum():,} / {boot_mask.sum():,} bootstrapped"
)
print(f"\nBootstrap summary:")
print(f"  Genes with bootstrap: {boot_mask.sum():,}")
print(f"  Min p_value_boot:     {dm['p_value_boot'].min():.4f}")
print(f"  Min p_value_chisq:    {dm.loc[converged_mask, 'p_value_chisq'].min():.2e}")

## Merge and filter


In [ ]:
# Filter to converged genes before merging
dm_conv = dm[converged_mask].copy()
print(f"Starting with {len(dm_conv):,} converged genes")

# Inner join - DirMult + Delta-psi
df = dm_conv.merge(dp, on="gene_id", suffixes=("_dm", "_dp"))
print(f"After DirMult ⨝ Delta-psi: {len(df):,} genes")

# Inner join - add ρ_W (gene-level)
df = df.merge(rho_w, on="gene_id", suffixes=("", "_rw"))
print(f"After ⨝ ρ_W:              {len(df):,} genes")

# Ensure padj column is unambiguous
if "padj_dm" in df.columns:
    df = df.rename(columns={"padj_dm": "padj"})

# Filter: non-NaN delta_psi
n_pre = len(df)
df = df[df["delta_psi"].notna()].copy()
print(f"After non-NaN delta_psi:    {len(df):,} genes  (removed {n_pre - len(df):,})")

# Filter: non-NaN rho_W
n_pre = len(df)
df = df[df["rho_W"].notna()].copy()
print(f"After non-NaN ρ_W:          {len(df):,} genes  (removed {n_pre - len(df):,})")

print(f"\nFiltered table: {df.shape}")

## ρ_W (gene-level concordance with pseudotime)

ρ_W is already gene-level - no per-site aggregation needed. Below: quick diagnostics on the merged table.


In [ ]:
print(f"|ρ_W| > 0.3:                       {(df['abs_rho_W'] > 0.3).sum():,}")
print(
    f"|ρ_W| > 0.3 & padj_BH_W < 0.05:    {((df['abs_rho_W'] > 0.3) & (df['padj_BH_W'] < 0.05)).sum():,}"
)

# Sign agreement between Δψ and ρ_W (independent diagnostic - both should agree on direction
# when there is a real monotone trend)
sign_agreement = np.sign(df["delta_psi"]) == np.sign(df["rho_W"])
print(
    f"\nSign(Δψ) == Sign(ρ_W):              {sign_agreement.sum():,} / {len(df):,}  ({sign_agreement.mean()*100:.1f}%)"
)
df["dpsi_rho_concordant"] = sign_agreement

## Compute optimized composite score


In [ ]:
# --- Significance: min-max scaled -log10(padj_chisq) ---
neglog10_padj = -np.log10(df["padj"].clip(lower=1e-300))
nlog_min = neglog10_padj.min()
nlog_max = neglog10_padj.max()
df["scaled_neglog10_padj"] = (neglog10_padj - nlog_min) / (nlog_max - nlog_min)

# --- Effect size: percentile rank of |Δψ| ---
df["rank_delta_psi"] = df["delta_psi"].abs().rank(pct=True)

# --- Concordance: percentile rank of |ρ_W| (gene-level, replaces previous mean-of-per-site) ---
df["rank_abs_rho_W"] = df["abs_rho_W"].rank(pct=True)

# --- Composite score (absolute) ---
df["composite_score_abs"] = (
    W_PADJ * df["scaled_neglog10_padj"]
    + W_DPSI * df["rank_delta_psi"]
    + W_RHO * df["rank_abs_rho_W"]
)

# --- Composite score (signed by delta_psi direction) ---
df["composite_score_signed"] = np.sign(df["delta_psi"]) * df["composite_score_abs"]

# Sort by absolute composite score (descending)
df = df.sort_values("composite_score_abs", ascending=False).reset_index(drop=True)

print(f"Weights: padj={W_PADJ}, delta_psi={W_DPSI}, abs_rho_W={W_RHO}")
print(f"\nComposite score (absolute) summary:")
print(df["composite_score_abs"].describe().to_string())
print(f"\nComposite score (signed) summary:")
print(df["composite_score_signed"].describe().to_string())

## Trajectory pattern classification


In [ ]:
BOUNDARY = 0.20  # pseudotime fraction considered "near boundary"


def classify_trajectory(row):
    dpsi = row["delta_psi"]
    t_min = row["t_at_min"]
    t_max = row["t_at_max"]

    if pd.isna(dpsi) or dpsi == 0:
        return "Stable"

    direction = "Lengthening" if dpsi > 0 else "Shortening"

    if pd.isna(t_min) or pd.isna(t_max):
        return direction

    min_near_start = t_min < BOUNDARY
    min_near_end = t_min > (1 - BOUNDARY)
    max_near_start = t_max < BOUNDARY
    max_near_end = t_max > (1 - BOUNDARY)

    if dpsi > 0:  # Lengthening: psi increases
        is_monotonic = min_near_start and max_near_end
    else:  # Shortening: psi decreases
        is_monotonic = max_near_start and min_near_end

    if is_monotonic:
        return direction
    else:
        return f"Complex|{direction}"


df["trajectory_pattern"] = df.apply(classify_trajectory, axis=1)
df["direction"] = np.where(df["delta_psi"] < 0, "Shortening", "Lengthening")

# Entropy direction: how the spread of UTR usage changes along pseudotime
#   < 0 -> Decreasing (concentrating on fewer sites)
#   > 0 -> Increasing (spreading across sites)
df["entropy_direction"] = "Unknown"
_h_valid = df["entropy_rho"].notna()
df.loc[_h_valid & (df["entropy_rho"] < 0), "entropy_direction"] = "Decreasing"
df.loc[_h_valid & (df["entropy_rho"] >= 0), "entropy_direction"] = "Increasing"

print("Trajectory pattern distribution:")
pattern_counts = df["trajectory_pattern"].value_counts()
for pattern, count in pattern_counts.items():
    print(f"  {pattern:25s}  {count:5,d}  ({count/len(df)*100:5.1f}%)")

print(
    "\nEntropy direction distribution (significant entropy_rho only, padj_BH_H < 0.05):"
)
_sig_h = df["padj_BH_H"] < 0.05
for d, count in df.loc[_sig_h, "entropy_direction"].value_counts().items():
    print(f"  {d:12s}  {count:5,d}")

In [ ]:
# Trajectory pattern visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Bar chart of trajectory patterns
ax = axes[0]
colors_pattern = {
    "Lengthening": "#B2182B",
    "Shortening": "#2166AC",
    "Complex|Lengthening": "#F4A582",
    "Complex|Shortening": "#92C5DE",
    "Stable": "#999999",
}
ordered_patterns = [
    "Shortening",
    "Complex|Shortening",
    "Stable",
    "Complex|Lengthening",
    "Lengthening",
]
counts = [pattern_counts.get(p, 0) for p in ordered_patterns]
bar_colors = [colors_pattern.get(p, "grey") for p in ordered_patterns]
bars = ax.barh(
    ordered_patterns, counts, color=bar_colors, edgecolor="white", linewidth=0.5
)
for bar, val in zip(bars, counts):
    ax.text(
        bar.get_width() + 20,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}",
        ha="left",
        va="center",
        fontsize=9,
    )
ax.set_xlabel("Number of genes")
ax.set_title("Trajectory pattern distribution")

# Middle: Distribution of signed composite scores
ax = axes[1]
ax.hist(
    df["composite_score_signed"],
    bins=80,
    color="#4878CF",
    edgecolor="white",
    linewidth=0.3,
)
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_xlabel("Signed composite score")
ax.set_ylabel("Number of genes")
ax.set_title("Distribution of signed composite scores\n(+) Lengthening  (−) Shortening")

# Right: entropy_rho distribution by trajectory pattern
# Captures concentration-vs-spread dynamics independent of the centroid shift
ax = axes[2]
for pattern in [
    "Shortening",
    "Lengthening",
    "Complex|Shortening",
    "Complex|Lengthening",
]:
    subset = df[df["trajectory_pattern"] == pattern]
    if len(subset) > 0 and subset["entropy_rho"].notna().any():
        ax.hist(
            subset["entropy_rho"].dropna(),
            bins=30,
            alpha=0.5,
            label=f"{pattern} (n={len(subset):,})",
            color=colors_pattern.get(pattern, "grey"),
        )
ax.axvline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("entropy_rho   (Spearman of per-cell H vs pseudotime)")
ax.set_ylabel("Number of genes")
ax.set_title(
    "Entropy dynamics by trajectory pattern\n(+) UTR usage SPREADING   (−) CONCENTRATING"
)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Method agreement diagnostics


In [ ]:
# Define significant gene sets
sig_dm = set(df.loc[df["padj"] < 0.05, "gene_id"])
q75_dpsi = df["delta_psi"].abs().quantile(0.75)
sig_dp = set(df.loc[df["delta_psi"].abs() >= q75_dpsi, "gene_id"])
sig_rw = set(df.loc[(df["padj_BH_W"] < 0.05) & (df["abs_rho_W"] > 0.2), "gene_id"])
sig_h = set(df.loc[(df["padj_BH_H"] < 0.05) & (df["abs_entropy_rho"] > 0.2), "gene_id"])

print(f"Total genes in ranking: {len(df):,}")
print(
    f"  DirMult significant (padj < 0.05):  {len(sig_dm):,}  ({len(sig_dm)/len(df)*100:.1f}%)"
)
print(
    f"  Delta-psi top 25% (|Δψ| ≥ {q75_dpsi:.4f}): {len(sig_dp):,}  ({len(sig_dp)/len(df)*100:.1f}%)"
)
print(
    f"  ρ_W significant (padj_BH_W < 0.05 & |ρ_W| > 0.2):           {len(sig_rw):,}  ({len(sig_rw)/len(df)*100:.1f}%)"
)
print(
    f"  entropy_rho significant (padj_BH_H < 0.05 & |entropy_rho|>0.2): {len(sig_h):,}  ({len(sig_h)/len(df)*100:.1f}%)"
)
print(
    f"  4-way overlap (DirMult ∩ Δψ ∩ ρ_W ∩ entropy):                    {len(sig_dm & sig_dp & sig_rw & sig_h):,}"
)

# Top genes
show_cols = [
    "gene_name",
    "gene_id",
    "composite_score_abs",
    "composite_score_signed",
    "trajectory_pattern",
    "entropy_direction",
    "dpsi_rho_concordant",
    "padj",
    "delta_psi",
    "nmi",
    "rho_W",
    "abs_rho_W",
    "padj_BH_W",
    "entropy_rho",
    "abs_entropy_rho",
    "padj_BH_H",
    "K",
    "n_cells",
]
show_cols = [c for c in show_cols if c in df.columns]

In [ ]:
# Top genes by directionality
print("— Top 15 LENGTHENING genes (highest signed composite score) —")
df_lengthening = df[df["delta_psi"] > 0].sort_values(
    "composite_score_signed", ascending=False
)
display(df_lengthening[show_cols].head(15))

print("\n— Top 15 SHORTENING genes (most negative signed composite score) —")
df_shortening = df[df["delta_psi"] < 0].sort_values(
    "composite_score_signed", ascending=True
)
display(df_shortening[show_cols].head(15))

In [ ]:
# Three-way rank correlation heatmap
rank_cols = ["scaled_neglog10_padj", "rank_delta_psi", "rank_abs_rho_W"]
corr = df[rank_cols].corr(method="spearman")
labels = ["DirMult\n-log$_{10}$(padj)", "|Δψ|", "|ρ_W|"]

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    corr,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    xticklabels=labels,
    yticklabels=labels,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Spearman correlation between composite components")
plt.tight_layout()
plt.show()

In [ ]:
# Venn diagram of significant gene sets (composite-relevant metrics)
fig, ax = plt.subplots(figsize=(7, 5))
venn3(
    [sig_dm, sig_dp, sig_rw],
    set_labels=(
        "DirMult\n(padj < 0.05)",
        "|Δψ| top 25%",
        "ρ_W\n(padj_BH_W < 0.05 & |ρ_W| > 0.2)",
    ),
    ax=ax,
)
ax.set_title("Gene set overlap — composite-ranking metrics")
plt.tight_layout()
plt.show()

# entropy_rho lives on a separate axis - show its overlap as a 2-set Venn
# against the union of the three primary metrics, since it captures
# orthogonal compositional information (concentration vs spread).
fig, ax = plt.subplots(figsize=(7, 5))
from matplotlib_venn import venn2

primary_union = sig_dm | sig_dp | sig_rw
venn2(
    [primary_union, sig_h],
    set_labels=(
        "Composite-relevant\n(DirMult ∪ Δψ ∪ ρ_W)",
        "entropy_rho\n(padj_BH_H < 0.05 & |entropy_rho| > 0.2)",
    ),
    ax=ax,
)
ax.set_title("entropy_rho hits vs composite-relevant gene set")
plt.tight_layout()
plt.show()

## Method 1 vs Method 2 comparison

Diagnostic: how the results change with the shared-s fix, the
conditional MLE refit, and the exclusion of failed fits from BH adjustment.


In [ ]:
# Load method 1 composite ranking (weighted version)
m1 = pd.read_csv(BASEPATH / "DirMult/composite_ranking_optimized_v2.csv")
print(f"Method 1 ranking: {m1.shape[0]:,} genes")
print(f"Method 2 ranking: {len(df):,} genes")

# Merge for comparison
comp = df[["gene_id", "gene_name", "composite_score_abs", "padj", "delta_psi"]].merge(
    m1[["gene_id", "composite_score_abs", "padj"]].rename(
        columns={"composite_score_abs": "cs_m1", "padj": "padj_m1"}
    ),
    on="gene_id",
    how="inner",
)
comp = comp.rename(columns={"composite_score_abs": "cs_m2", "padj": "padj_m2"})
comp["rank_m2"] = comp["cs_m2"].rank(ascending=False).astype(int)
comp["rank_m1"] = comp["cs_m1"].rank(ascending=False).astype(int)
comp["rank_change"] = comp["rank_m1"] - comp["rank_m2"]

rho_overall, pval_overall = stats.spearmanr(comp["rank_m1"], comp["rank_m2"])
print(f"\nOverlapping genes: {len(comp):,}")
print(
    f"Rank correlation (M1 vs M2): Spearman ρ = {rho_overall:.4f}, p = {pval_overall:.2e}"
)

# Discovery overlap
sig_m1 = set(comp.loc[comp["padj_m1"] < 0.05, "gene_id"])
sig_m2 = set(comp.loc[comp["padj_m2"] < 0.05, "gene_id"])
print(f"\nDiscovery overlap (padj < 0.05):")
print(f"  Method 1 only: {len(sig_m1 - sig_m2):,}")
print(f"  Method 2 only: {len(sig_m2 - sig_m1):,}")
print(f"  Both:          {len(sig_m1 & sig_m2):,}")
print(f"  Jaccard index: {len(sig_m1 & sig_m2) / len(sig_m1 | sig_m2):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Left: Rank scatter
ax = axes[0]
ax.scatter(
    comp["rank_m1"],
    comp["rank_m2"],
    s=4,
    alpha=0.15,
    rasterized=True,
    color="#4878CF",
)
ax.plot([0, len(comp)], [0, len(comp)], "k--", lw=0.8, alpha=0.5)
ax.set_xlabel("Rank (Method 1)")
ax.set_ylabel("Rank (Method 2)")
ax.set_title(f"Rank comparison — Spearman ρ = {rho_overall:.3f}")

# Middle: Discovery overlap Venn
ax = axes[1]
venn2(
    [sig_m1, sig_m2],
    set_labels=("Method 1\n(padj < 0.05)", "Method 2\n(padj_chisq < 0.05)"),
    ax=ax,
)
ax.set_title("Discovery overlap")

# Right: p-value comparison scatter
ax = axes[2]
ax.scatter(
    -np.log10(comp["padj_m1"].clip(lower=1e-300)),
    -np.log10(comp["padj_m2"].clip(lower=1e-300)),
    s=4,
    alpha=0.15,
    rasterized=True,
    color="#4878CF",
)
max_val = max(
    -np.log10(comp["padj_m1"].clip(lower=1e-300)).max(),
    -np.log10(comp["padj_m2"].clip(lower=1e-300)).max(),
)
ax.plot([0, max_val], [0, max_val], "k--", lw=0.8, alpha=0.5)
ax.set_xlabel("-log10(padj) Method 1")
ax.set_ylabel("-log10(padj_chisq) Method 2")
ax.set_title("p-value concordance")

plt.tight_layout()
plt.show()

In [ ]:
# Top rank gainers and losers
comp_sorted = comp.reindex(comp["rank_change"].abs().sort_values(ascending=False).index)

print("— Top 10 rank GAINERS (rose in Method 2) —")
gainers = comp.nlargest(10, "rank_change")
display(
    gainers[
        [
            "gene_name",
            "gene_id",
            "rank_m1",
            "rank_m2",
            "rank_change",
            "padj_m1",
            "padj_m2",
            "delta_psi",
        ]
    ]
)

print("\n— Top 10 rank LOSERS (dropped in Method 2) —")
losers = comp.nsmallest(10, "rank_change")
display(
    losers[
        [
            "gene_name",
            "gene_id",
            "rank_m1",
            "rank_m2",
            "rank_change",
            "padj_m1",
            "padj_m2",
            "delta_psi",
        ]
    ]
)

## Bootstrap vs chi-squared p-value comparison


In [ ]:
# For genes with both p-values, compare chisq vs bootstrap
boot_genes = df[df["padj_boot"].notna()].copy()
print(f"Genes with both chisq and bootstrap padj: {len(boot_genes):,}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: padj scatter
ax = axes[0]
x = -np.log10(boot_genes["padj"].clip(lower=1e-300))
y = -np.log10(boot_genes["padj_boot"].clip(lower=1e-300))
ax.scatter(x, y, s=6, alpha=0.3, rasterized=True, color="#4878CF")
max_val = max(x.max(), y.max())
ax.plot([0, max_val], [0, max_val], "k--", lw=0.8, alpha=0.5)
# Annotate the bootstrap floor
boot_floor = -np.log10(0.005)
ax.axhline(boot_floor, color="#B2182B", ls=":", lw=1, alpha=0.7)
ax.text(
    max_val * 0.6,
    boot_floor + 0.1,
    "Bootstrap floor (p=0.005)",
    fontsize=8,
    color="#B2182B",
)
ax.set_xlabel("-log10(padj_chisq)")
ax.set_ylabel("-log10(padj_boot)")
ax.set_title(f"Chi-sq vs Bootstrap padj (n={len(boot_genes):,})")

# Right: agreement classification
ax = axes[1]
both_sig = ((boot_genes["padj"] < 0.05) & (boot_genes["padj_boot"] < 0.05)).sum()
chisq_only = ((boot_genes["padj"] < 0.05) & (boot_genes["padj_boot"] >= 0.05)).sum()
boot_only = ((boot_genes["padj"] >= 0.05) & (boot_genes["padj_boot"] < 0.05)).sum()
neither = ((boot_genes["padj"] >= 0.05) & (boot_genes["padj_boot"] >= 0.05)).sum()

labels = ["Both sig", "Chisq only", "Boot only", "Neither"]
values = [both_sig, chisq_only, boot_only, neither]
colors = ["#2166AC", "#F4A582", "#92C5DE", "#D9D9D9"]
bars = ax.bar(labels, values, color=colors, edgecolor="white")
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        f"{val:,}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("Number of genes")
ax.set_title("padj_chisq vs padj_boot agreement\n(among bootstrapped genes)")

plt.tight_layout()
plt.show()

print(
    f"\nAgreement: {both_sig + neither:,} / {len(boot_genes):,} ({(both_sig + neither)/len(boot_genes)*100:.1f}%)"
)
print(
    f"Discordant: {chisq_only + boot_only:,} ({(chisq_only + boot_only)/len(boot_genes)*100:.1f}%)"
)

## Non-converged gene diagnostic

507 genes did not converge. Check whether they differ systematically (K, number of cells, UMIs).


In [ ]:
nonconv = dm[~converged_mask].copy()
conv = dm[converged_mask].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# K distribution
ax = axes[0]
for label, subset, color in [
    ("Converged", conv, "#4878CF"),
    ("Non-conv", nonconv, "#B2182B"),
]:
    ax.hist(
        subset["K"],
        bins=range(2, subset["K"].max() + 2),
        alpha=0.5,
        color=color,
        label=f"{label} (n={len(subset):,})",
        density=True,
        edgecolor="white",
    )
ax.set_xlabel("K (number of isoforms)")
ax.set_ylabel("Density")
ax.set_title("K distribution")
ax.legend(fontsize=9)

# n_cells distribution
ax = axes[1]
for label, subset, color in [
    ("Converged", conv, "#4878CF"),
    ("Non-conv", nonconv, "#B2182B"),
]:
    ax.hist(
        np.log10(subset["n_cells"].clip(lower=1)),
        bins=40,
        alpha=0.5,
        color=color,
        label=label,
        density=True,
        edgecolor="white",
    )
ax.set_xlabel("log10(n_cells)")
ax.set_ylabel("Density")
ax.set_title("Cell count distribution")
ax.legend(fontsize=9)

# UMI distribution
ax = axes[2]
for label, subset, color in [
    ("Converged", conv, "#4878CF"),
    ("Non-conv", nonconv, "#B2182B"),
]:
    ax.hist(
        np.log10(subset["UMIs"].clip(lower=1)),
        bins=40,
        alpha=0.5,
        color=color,
        label=label,
        density=True,
        edgecolor="white",
    )
ax.set_xlabel("log10(UMIs)")
ax.set_ylabel("Density")
ax.set_title("UMI distribution")
ax.legend(fontsize=9)

plt.suptitle("Non-converged gene characteristics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(
    f"\nMedian K:      Converged={conv['K'].median():.0f}  Non-conv={nonconv['K'].median():.0f}"
)
print(
    f"Median n_cells: Converged={conv['n_cells'].median():.0f}  Non-conv={nonconv['n_cells'].median():.0f}"
)
print(
    f"Median UMIs:    Converged={conv['UMIs'].median():.0f}  Non-conv={nonconv['UMIs'].median():.0f}"
)

## Save composite ranking


In [ ]:
# Add weight metadata columns for reproducibility
df["composite_weight_padj"] = W_PADJ
df["composite_weight_dpsi"] = W_DPSI
df["composite_weight_rho"] = W_RHO

out_path = BASEPATH / "DirMult_optimized/composite_ranking_optimized2_df5.csv"
df.to_csv(out_path, index=False)
print(f"Saved composite ranking to:")
print(f"  {out_path}")
print(f"  Rows: {len(df):,}  |  Columns: {len(df.columns)}")

# Key columns summary
key_cols = [
    "gene_id",
    "gene_name",
    "padj",
    "padj_mixed",
    "padj_boot",
    "p_value_chisq",
    "p_value_boot",
    "delta_psi",
    "delta_psi_max",
    "nmi",
    "rho_W",
    "abs_rho_W",
    "padj_BH_W",
    "entropy_rho",
    "abs_entropy_rho",
    "padj_BH_H",
    "entropy_direction",
    "dpsi_rho_concordant",
    "composite_score_abs",
    "composite_score_signed",
    "direction",
    "trajectory_pattern",
    "K",
    "UMIs",
    "n_cells",
    "converged",
    "cells_per_param",
    "max_var_ratio",
    "min_var_ratio",
    "scaled_neglog10_padj",
    "rank_delta_psi",
    "rank_abs_rho_W",
]
present = [c for c in key_cols if c in df.columns]
missing = [c for c in key_cols if c not in df.columns]
print(f"\nKey columns present: {len(present)}/{len(key_cols)}")
if missing:
    print(f"Missing: {missing}")